In [1]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

I0000 00:00:1774204356.718551   50348 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774204356.774778   50348 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774204358.005040   50348 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:

# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [3]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

I0000 00:00:1774204619.889561   50348 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [4]:
model.get_weights()

[array([[ 0.02320291, -0.03954391, -0.07054337, ..., -0.01848427,
         -0.03698714, -0.01191367],
        [-0.02663178,  0.023744  ,  0.04261795, ..., -0.00535111,
          0.02347347, -0.05048207],
        [ 0.00580134, -0.01797256,  0.01610337, ...,  0.00276372,
          0.04131193,  0.02247365],
        ...,
        [-0.0443215 ,  0.03391702,  0.00560515, ..., -0.01825414,
          0.0402156 , -0.00792581],
        [ 0.04343471,  0.00659766, -0.03837803, ...,  0.01765276,
         -0.07062893,  0.07422762],
        [ 0.01879011,  0.05922397, -0.02563115, ...,  0.0011002 ,
          0.02619459,  0.04104916]], shape=(10000, 128), dtype=float32),
 array([[ 0.16453256, -0.06977958, -0.1546225 , ..., -0.11353725,
         -0.150638  , -0.110764  ],
        [ 0.09266616,  0.06776041,  0.07272897, ...,  0.0382832 ,
         -0.13173515,  0.0704212 ],
        [-0.11095432,  0.11365657,  0.03148187, ..., -0.1933093 ,
          0.07371844,  0.1551957 ],
        ...,
        [-0.1069683

In [5]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [6]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]



In [7]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

I0000 00:00:1774204707.404813   51187 service.cc:153] XLA service 0x62e4126cda10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774204707.404857   51187 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.2.0; Runtime: 12.3.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1774204707.437583   51187 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774204707.481414   51187 cuda_dnn.cc:461] Loaded cuDNN version 92000


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 587ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.8593432307243347


I0000 00:00:1774204707.836081   51187 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
